<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Modulo 2</h2><br/>
<h1>Semana 11 · Viernes — Explicabilidad de Modelos</h1>
<h3>SHAP + LIME + Feature Importance sobre LightGBM de Telco Churn</h3>
<br/>
    <b>Instructor:</b> Jesus Ortiz · SkillNest
</div>
<br/>

Hoy abrimos la "caja negra". Hasta ahora nuestros modelos predecian — pero no nos decian POR QUE. En el trabajo real, eso no alcanza. Si vas a recomendar al gerente comercial que llame a 1000 clientes, te van a preguntar: ¿por que estos 1000? Si no podes responder, el modelo no entra en produccion.

Esta clase enseña las 3 tecnicas que mas se usan en la industria para explicar predicciones: Feature Importance, SHAP y LIME. Cierra con un ejercicio sobre el modelo de Telco Churn de la semana 8.

## Objetivos

Al final de la clase van a poder:

1. Distinguir entre **explicabilidad global** (que aprende el modelo en general) y **local** (por que predijo ESTO para ESTE caso).
2. Calcular y leer **Feature Importance** built-in (gain/split) y **Permutation Importance**.
3. Usar **SHAP**: summary plot, dependence plot, force plot y waterfall plot.
4. Usar **LIME** para explicar predicciones individuales caso a caso.
5. Hacer **Partial Dependence Plots** para entender el efecto de una feature aislada.
6. Decidir QUE tecnica usar segun el caso de negocio.
7. Resolver un ejercicio donde explican a un cliente especifico por que se va a churnar.

# 1. Por que explicar modelos?

Imagina esta conversacion en el banco:

> **Cliente**: "Por que me negaron el credito?"  
> **Banco**: "Porque el modelo dijo que si."  
> **Cliente**: "Que modelo?"  
> **Banco**: "Random Forest con 500 arboles."  
> **Cliente**: "Pero por que YO?"  
> **Banco**: "..."

Sin explicabilidad, **el modelo no entra a produccion** en industrias reguladas (banca, salud, seguros). Y aunque NO sea regulado, sin poder explicar:

- El area comercial no confia en el modelo y no lo usa.
- Cuando el modelo falla, no podes diagnosticar por que.
- Los clientes no aceptan decisiones automatizadas sin justificacion.
- La gerencia no aprueba presupuesto para escalarlo.

**Explicabilidad NO es un nice-to-have. Es lo que separa un modelo de juguete de uno productivo.**

## Tipos de explicabilidad

| Tipo | Que responde | Ejemplo |
|---|---|---|
| **Global** | Que features importan MAS en general | "El tenure es el mejor predictor de churn en todo el dataset" |
| **Local** | Por que el modelo predijo X para ESTE caso | "Para el cliente #1234 churneara por su contrato mes-a-mes + fibra optica" |
| **Model-agnostic** | Funciona con CUALQUIER modelo (caja negra) | SHAP, LIME, Permutation Importance |
| **Model-specific** | Aprovecha la estructura del modelo | Feature Importance de RandomForest/LightGBM (built-in) |

Hoy vemos las 4. Vamos por el dataset de Telco Churn de la semana 8.

# 2. Setup: cargar modelo entrenado de Telco Churn

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import lightgbm as lgb

import shap
import lime
import lime.lime_tabular

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', None)

URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(URL).drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])

for c in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df[c] = (df[c] == 'Yes').astype(int)
df['gender'] = (df['gender'] == 'Male').astype(int)
df['Contract'] = df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})

y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['Churn'])

NUM_STD = ['tenure', 'MonthlyCharges']
NUM_MM = ['TotalCharges']
NOM = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']
PASS = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Contract']

prep = ColumnTransformer([
    ('std', StandardScaler(), NUM_STD),
    ('mm', MinMaxScaler(), NUM_MM),
    ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), NOM),
    ('pass', 'passthrough', PASS),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_p = prep.fit_transform(X_train)
X_test_p = prep.transform(X_test)

# Nombres de features post-OneHot
feature_names = (NUM_STD + NUM_MM +
                 list(prep.named_transformers_['ohe'].get_feature_names_out(NOM)) +
                 PASS)

modelo = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6, num_leaves=31,
    random_state=42, verbose=-1
).fit(X_train_p, y_train)

print(f'Modelo entrenado: LightGBM')
print(f'Accuracy en test: {modelo.score(X_test_p, y_test):.4f}')
print(f'Features post-preprocesamiento: {len(feature_names)}')

# 3. Feature Importance built-in (rapido pero limitado)

Los modelos basados en arboles (Random Forest, LightGBM, LightGBM) traen feature importance "de fabrica". Hay 3 tipos:

- **Gain**: cuanto reduce el error cada feature (la mejor para interpretar el "valor").
- **Split (Weight)**: cuantas veces se uso cada feature en los splits (puede inflar features con muchos valores unicos).
- **Cover**: cuantas observaciones cubre cada split (mide la "amplitud").

**Limitaciones:**
- No es comparable entre modelos distintos (cada uno calcula diferente).
- Puede ser engañoso si hay correlaciones entre features.
- Solo dice "que feature importa", no DIRECCION (positivo/negativo).

In [ ]:
# Feature importance de XGBoost (por gain)
importances_gain = dict(zip(feature_names, modelo.booster_.feature_importance(importance_type='gain')))

imp_df = pd.DataFrame(importances_gain.items(), columns=['feature', 'gain'])
imp_df = imp_df.sort_values('gain', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(imp_df['feature'][::-1], imp_df['gain'][::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Gain (reduccion de error promedio por uso)')
ax.set_title('Top 15 features por GAIN (XGBoost built-in)', fontweight='bold')
plt.tight_layout(); plt.show()
print(imp_df.head(10).to_string(index=False))

**Lo que veo en feature importance built-in**

Las features mas importantes (segun LightGBM) son `tenure`, `MonthlyCharges`, `Contract` y `OnlineSecurity_Yes`. Esto coincide con lo que vimos en el EDA: clientes nuevos con contrato mes-a-mes y sin servicios adicionales tienen mas riesgo.

**Pero esto tiene un problema:** solo dice "que importa", NO dice si es para CHURN o NO CHURN. Por ejemplo, `tenure` aparece como la mas importante — pero no sabemos si valores ALTOS de tenure aumentan o bajan el churn. Para eso necesitamos SHAP.

# 4. Permutation Importance (model-agnostic)

Esta tecnica funciona con CUALQUIER modelo (no solo arboles). La idea es:

1. Entrenas el modelo.
2. Mides el score base (ej: accuracy).
3. Para cada feature: **mezclas sus valores aleatoriamente** y vuelves a medir.
4. La importancia = cuanto BAJA el score por mezclar esa feature.

Si mezclar una feature destroza el modelo → era importante.  
Si mezclarla no cambia nada → no aportaba.

**Ventaja sobre built-in:** funciona con LinearRegression, SVM, redes, ensemble — todo. Y es mas honesta porque mide el impacto real en performance.

In [ ]:
# Permutation Importance (puede tardar 30 seg)
result = permutation_importance(
    modelo, X_test_p, y_test,
    n_repeats=10, random_state=42, n_jobs=-1, scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature': feature_names,
    'mean_drop': result.importances_mean,
    'std': result.importances_std,
}).sort_values('mean_drop', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(perm_df['feature'][::-1], perm_df['mean_drop'][::-1],
        xerr=perm_df['std'][::-1], color='#70AD47', edgecolor='white')
ax.set_xlabel('Caida en AUC-ROC al mezclar esta feature')
ax.set_title('Top 15 features por Permutation Importance (model-agnostic)', fontweight='bold')
plt.tight_layout(); plt.show()
print(perm_df.head(10)[['feature', 'mean_drop']].to_string(index=False))

**Lo que veo en permutation importance**

Los resultados son similares pero NO identicos a feature importance built-in. Algunas diferencias:

- Las **barras de error** (std) muestran cuanta varianza tiene cada estimacion. Features con error grande son menos confiables.
- Permutation puede dar valores **negativos** (raro): significa que mezclar esa feature MEJORO el score → el modelo la estaba usando como ruido.
- Es mas LENTA pero mas confiable.

**Regla practica:** si built-in y permutation coinciden en el top 5-10, podes confiar en ambas. Si difieren mucho, hay correlaciones entre features que estan confundiendo al built-in.

# 5. SHAP — el estandar de la industria

**SHAP (SHapley Additive exPlanations)** es la tecnica mas usada hoy en explicabilidad. Combina lo mejor de:
- **Global**: te dice que features importan en general.
- **Local**: te dice POR QUE el modelo predijo ESTO para ESTE caso.
- **Direccion**: te dice si la feature EMPUJA hacia 1 o hacia 0.

Esta basada en teoria de juegos: los **Shapley values** miden la contribucion "justa" de cada jugador (feature) a un resultado (prediccion).

**Por que SHAP es el estandar:**
- Tiene garantias matematicas (consistencia, eficiencia).
- Funciona con CUALQUIER modelo.
- Para LightGBM/LightGBM hay un algoritmo optimizado (TreeSHAP) que es rapido.
- Genera visualizaciones muy claras.

In [ ]:
# Crear el explainer (rapido para XGBoost via TreeSHAP)
explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_test_p)
print(f'shap_values shape: {shap_values.shape}')
print(f'  filas = clientes ({X_test_p.shape[0]})')
print(f'  cols  = features ({len(feature_names)})')
print(f'  cada celda = cuanto empuja esa feature hacia churn (positivo) o no churn (negativo)')

## 5.1 Summary plot — vista global

El summary plot muestra DOS cosas a la vez:
- En el eje Y: features ordenadas por importancia global.
- Cada punto: un cliente.
- Color del punto: valor de la feature (rojo = alto, azul = bajo).
- Posicion X: cuanto empuja esa feature al modelo (derecha = mas churn).

In [ ]:
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_p, feature_names=feature_names, max_display=15, show=False)
plt.tight_layout(); plt.show()

**Como leer este grafico**

- **`tenure`** (arriba = mas importante): valores BAJOS (azul) empujan A LA DERECHA (mas churn). Valores ALTOS (rojo) empujan A LA IZQUIERDA (menos churn). Conclusion: clientes nuevos churnan mas, clientes antiguos se quedan.
- **`Contract`** (valor 0=mes a mes, 1=anual, 2=2 anos): valores bajos (mes a mes) empujan a churn. Valores altos (2 anos) protegen.
- **`MonthlyCharges`**: valores ALTOS (rojo) empujan a churn. Los planes caros pierden mas clientes.
- **`OnlineSecurity_Yes`**: TENER el servicio (valor 1) empuja contra churn. NO tenerlo aumenta churn.

Esta es la magia de SHAP: no solo dice "tenure importa", dice "tenure bajo CAUSA churn". Es informacion accionable.

## 5.2 Dependence plot — efecto de UNA feature en detalle

Muestra como la prediccion cambia con el valor de una feature, y colorea por una segunda feature para detectar interacciones.

In [ ]:
# Tenure es la mas importante, vamos a ver su efecto en detalle
tenure_idx = feature_names.index('tenure')
fig, ax = plt.subplots(figsize=(10, 5))
shap.dependence_plot(tenure_idx, shap_values, X_test_p, feature_names=feature_names,
                     show=False, ax=ax)
plt.title('SHAP dependence: tenure', fontweight='bold')
plt.tight_layout(); plt.show()

**Lo que veo en el dependence plot de tenure**

Curva clara descendente: a mas tenure, menor probabilidad de churn (SHAP value negativo).

El "umbral critico" parece estar entre tenure ~12 y tenure ~30:
- Antes de 12 meses: alto riesgo (SHAP positivo, empuja a churn).
- Entre 12 y 30: zona ambigua, baja gradualmente.
- Despues de 30 meses: el cliente esta "fidelizado", SHAP claramente negativo.

**Insight de negocio:** las campanas de retencion deben enfocarse en los primeros 12 meses de vida del cliente. Despues de los 30 meses, la inversion en retencion no rinde tanto (ya estan fidelizados).

## 5.3 Force plot y Waterfall — explicacion LOCAL de UN cliente

Aca SHAP brilla: explica POR QUE el modelo predijo X para UN cliente especifico. Esto es lo que le mostrarian a un cliente que pregunta "por que me dijeron que iba a churnar?".

In [ ]:
# Eligir el cliente con mayor probabilidad de churn en el test set
probas = modelo.predict_proba(X_test_p)[:, 1]
idx_mas_riesgoso = int(np.argmax(probas))
print(f'Cliente test #{idx_mas_riesgoso}:')
print(f'  Probabilidad de churn segun el modelo: {probas[idx_mas_riesgoso]:.4f}')
print(f'  Real: {"Si churn" if y_test.iloc[idx_mas_riesgoso] == 1 else "No churn"}')
print()

# Datos crudos del cliente
cliente_data = X_test.iloc[idx_mas_riesgoso]
print('Sus datos:')
for k, v in cliente_data.items():
    print(f'  {k:25s} {v}')

# Waterfall plot
expected_value = explainer.expected_value
if hasattr(expected_value, '__len__'):
    expected_value = expected_value[0] if len(expected_value) == 1 else expected_value

shap_exp = shap.Explanation(
    values=shap_values[idx_mas_riesgoso],
    base_values=expected_value,
    data=X_test_p[idx_mas_riesgoso],
    feature_names=feature_names
)
fig = plt.figure(figsize=(10, 7))
shap.plots.waterfall(shap_exp, max_display=12, show=False)
plt.tight_layout(); plt.show()

**Como leer el waterfall plot**

Empezamos desde abajo (E[f(x)] = el valor esperado, es el "promedio" del modelo). De ahi cada feature SUMA o RESTA contribuciones:

- Barras ROJAS empujan hacia churn (suma probabilidad).
- Barras AZULES empujan contra churn (resta probabilidad).

El cliente termina con f(x) = la probabilidad final.

**Para este cliente:**
- Tenure muy bajo (probable < 12 meses): +0.X de contribucion al churn.
- Contract = mes a mes: +0.Y al churn.
- TotalCharges bajo (sigue del tenure): +0.Z.
- MonthlyCharges alto: +0.W.

Esto es lo que muestras al area comercial: **"Este cliente va a churnar por tenure bajo + contrato mes a mes + cargos altos. La campana de retencion deberia ofrecerle migracion a contrato anual con descuento."**

# 6. LIME — explicacion local con un enfoque distinto

**LIME (Local Interpretable Model-agnostic Explanations)** tambien explica predicciones individuales, pero con otra logica:

1. Toma el caso a explicar.
2. Genera muestras perturbadas alrededor de ese caso.
3. Entrena un modelo SIMPLE (regresion lineal) localmente sobre esas muestras.
4. Usa los coeficientes del modelo simple como explicacion.

**Diferencias con SHAP:**
- Mas rapido para casos individuales (no requiere entrenar un explainer global).
- Mas intuitivo de leer (es un modelo lineal simple).
- Menos garantias matematicas que SHAP.
- Puede dar resultados distintos entre ejecuciones (depende del muestreo aleatorio).

En la practica: **SHAP para reportes formales y para auditoria. LIME para debugging rapido caso a caso.**

In [ ]:
# LIME explainer
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train_p,
    feature_names=feature_names,
    class_names=['No Churn', 'Churn'],
    mode='classification'
)

# Explicar el mismo cliente que usamos con SHAP
exp = explainer_lime.explain_instance(
    X_test_p[idx_mas_riesgoso],
    modelo.predict_proba,
    num_features=10
)

# Mostrar como tabla
print(f'LIME explanation para cliente test #{idx_mas_riesgoso}:')
print(f'Probabilidad predicha: {probas[idx_mas_riesgoso]:.4f}')
print()
for feature, weight in exp.as_list():
    direccion = '↑ churn' if weight > 0 else '↓ churn'
    print(f'  {direccion}  {abs(weight):.4f}  |  {feature}')

**Como leer LIME**

Cada linea muestra una "condicion" que LIME identifico como relevante. Por ejemplo:
- `↑ churn  0.08  |  Contract <= 0.5` significa: "que Contract sea menor o igual a 0.5 (es decir, mes a mes) suma 0.08 a la probabilidad de churn".

LIME crea estas condiciones binarizando las features y entrenando una regresion lineal local. Es mas granular que SHAP en algunas dimensiones, pero **menos consistente entre ejecuciones**.

**Mi recomendacion:** usa LIME cuando quieras una explicacion rapida y conversacional. Usa SHAP cuando necesites rigor matematico o quieras analizar muchos casos en bulk.

# 7. Partial Dependence Plots (PDP)

Los PDP muestran el efecto MARGINAL de una feature sobre la prediccion, manteniendo las demas constantes (en promedio).

Es lo que un analista de negocio quiere ver: "si subo el MonthlyCharges en 10 dolares, cuanto sube el churn?".

In [ ]:
# PDP de las top 4 features
top_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract']
top_idx = [feature_names.index(f) for f in top_features]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
PartialDependenceDisplay.from_estimator(
    modelo, X_test_p, top_idx,
    feature_names=feature_names,
    ax=axes.flatten()[:len(top_idx)],
    line_kw={'color': '#C0504D', 'linewidth': 2}
)
plt.suptitle('Partial Dependence Plots - Top 4 features', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

**Como leer los PDPs**

Cada curva muestra como cambia la probabilidad de churn al mover el valor de UNA feature, promediando el efecto de las demas.

- **tenure**: caida fuerte en los primeros 12 meses, despues curva suave hacia abajo.
- **MonthlyCharges**: subida constante. A mas cargo mensual, mas churn.
- **TotalCharges**: curva descendente. Clientes con mucho total acumulado son los antiguos = menos churn.
- **Contract**: escalon brusco entre mes-a-mes (0) y 1-2 anos. Cada salto de tier reduce churn significativamente.

**Limitacion del PDP:** asume que las features son independientes. Si dos features estan correlacionadas (ej: tenure y TotalCharges), el PDP puede ser engañoso para combinaciones imposibles. SHAP es mas robusto en ese sentido.

---
# Ejercicio integrador

Trabajan como Data Scientists en la Telco. El gerente comercial les pide ayuda para entender 2 cosas:

## Reto

**Parte A — Explicacion global (15 min)**
1. Entrenar el mismo LightGBM con los datos de Telco Churn (esta hecho en la celda de setup).
2. Generar el SHAP summary plot.
3. Escribir 3 lineas: cuales son las 3 features mas predictivas y QUE direccion tienen (positivo a churn / negativo a churn).

**Parte B — Caso especifico (20 min)**
4. Identificar 2 clientes:
   - El de mayor probabilidad de churn (lo hicimos en la demo).
   - Un cliente con probabilidad MEDIA (entre 0.40 y 0.60).
5. Para cada uno: mostrar el waterfall plot de SHAP + la explicacion de LIME.
6. Comparar: para el caso medio, SHAP y LIME coinciden o difieren? Por que?

**Parte C — PDP enfocado (10 min)**
7. Generar el PDP de la feature `MonthlyCharges`.
8. Identificar el "punto de inflexion" donde el churn empieza a subir mas rapido.
9. Que recomendarian al area de pricing?

**Parte D — Recomendacion de negocio (15 min)**
Imagina que el banco te pide explicar tu modelo al area de Compliance (auditoria). Escriban 1 pagina en markdown defendiendo:

- Por que el modelo NO es discriminatorio (revisar SHAP de gender, SeniorCitizen, etc).
- Que features SI estan usando para predecir y por que tiene sentido logico.
- Que caso de uso especifico recomendarian usar el modelo PARA y cual NO.
- Como auditarian el modelo en produccion cada mes.

### Bonus (puntos extra)

- Probar `shap.plots.bar(shap_values)` para un grafico mas limpio para reportes.
- Probar `shap.plots.beeswarm()` y comparar contra summary_plot.
- Calcular el Permutation Importance pero con `scoring='f1'` en vez de `'roc_auc'`. Cambian las features mas importantes? Por que?

In [ ]:
# Parte A — SHAP summary



In [ ]:
# Parte B — Casos especificos (alto y medio riesgo)



In [ ]:
# Parte C — PDP de MonthlyCharges



# Parte D — Defensa ante Compliance (escribir en markdown)



## Cierre

Lo importante de hoy:

- **Feature Importance built-in**: rapido y bueno como primera mirada, pero no dice direccion.
- **Permutation Importance**: mas honesto, funciona con cualquier modelo.
- **SHAP**: el estandar de la industria. Global y local. Direccion clara. Visualizaciones potentes.
- **LIME**: alternativa rapida para casos individuales conversacionales.
- **PDP**: para visualizar el efecto de UNA feature en aislamiento.

**Regla de oro:** un modelo que no podes explicar, no podes defender. Y un modelo que no podes defender, no entra en produccion.

La proxima vez que entreguen un modelo, agreguen una seccion de **"interpretacion de las predicciones"** con SHAP. Eso los va a diferenciar como senior.

Con esto cerramos el modulo de Machine Learning. La proxima fase: proyectos finales y graduacion.